# Predicting Content Decay: A Machine Learning Approach to Editorial Triage
    
**Abstract:** As search engine traffic scales, maintaining content freshness across massive portfolios becomes manually impossible. This research asks whether historical search performance and content metadata can reliably predict future traffic decay, enabling automated editorial triage. We utilized a 79-million-row production data warehouse to engineer decay signals and trained a Random Forest classifier. By employing an honest, client-grouped validation design, we eliminated target leakage and client-memorization traps, achieving a robust 66% Precision@50—vastly outperforming the heuristic baseline of 30%. The resulting model successfully generated an automated editorial action playbook, proving that machine learning can accurately direct high-ROI content refreshes at scale.

## 1. Introduction & Problem Statement

Organic search traffic decays over time as content becomes stale, competitors publish fresher answers, and search intent evolves. Traditional editorial teams rely on manual audits or blunt heuristic rules to decide which pages to update. These methods are inefficient, resulting in wasted effort on evergreen content and missed opportunities on decaying high-value pages. 

Our objective was to build a predictive model that identifies pages at high risk of traffic loss before the loss compounds, effectively prioritizing editorial resources to maximize traffic retention.

## 2. Data

This research leverages the `internship-warehouse` (v20260703) release. 

- **Primary Source:** `fact_content_daily_performance` containing ~79M rows of daily search performance across 104 pseudonymized clients.
- **Dimensional Data:** `dim_content` containing metadata for 519,606 content items (e.g., word count, creation dates).
- **Date Windows:** We used the `month=2026-03` partition for training and ablation testing to ensure a sealed timeline, and the `month=2026-06` partition as our out-of-time sample to generate the final action playbook.
- **Exclusions:** We strictly filtered rows with `< 50 early impressions` to remove statistical noise from zero-traffic pages. We also excluded `trend_pct` and `is_declining_label` from the starter dataset to prevent fatal target leakage. All client IDs and content IDs were treated as opaque pseudonyms to ensure public safety and data privacy.

## 3. Methodology

### Feature Engineering
We engineered temporal and engagement features natively in DuckDB:
- `early_imps`, `early_clicks`, `early_pos` (Performance metrics prior to the label window)
- `early_ctr` (Calculated as clicks / impressions)
- `content_age_days` (Mathematical difference between creation date and the snapshot date)
- `word_count` 

### Label Definition
A page is defined as "declining" if its impressions in the second half of the month (`late_imps`) fell below 80% of its impressions in the first half of the month (`early_imps`). 

### Validation Design & Leakage Checks
A naive random split allows the model to memorize client-specific traffic baselines, artificially inflating scores. We utilized `GroupShuffleSplit` on `client_hash_id` to enforce a rigorous boundary, ensuring the model learned generalizable decay signals rather than client IDs. Furthermore, we audited our features for target leakage; including `late_imps` in the feature set falsely drove the model to 100% precision (the "Time Machine Trap"), which we rigorously stripped out.

### Baseline
Our heuristic baseline flagged pages ranking on Page 1 (avg_position <= 10) with a poor CTR (< 1%).

## 4. Results

We conducted an ablation test using Logistic Regression, Random Forest, and XGBoost to prove non-linear complexity. 

### Model vs. Baseline (Precision@50)
- **Heuristic Baseline:** 30.0%
- **Logistic Regression:** 50.0%
- **XGBoost:** 42.0%
- **Random Forest:** **66.0%**

Random Forest emerged as the superior model, effectively handling the extreme outliers inherent in SEO traffic data without overfitting.

### Feature Importances
The model leaned heavily on `content_age_days` (0.26) and `early_ctr` (0.26). This mathematically proves that content decay is a function of both *time* (staleness) and *relevance* (user engagement).

### Validation Audit
Our methodology checks confirmed that naive approaches yield dangerously inflated scores:
- **Naive Random Split Precision:** 82.0%
- **Honest Grouped Split Precision:** 80.0% (The model generalizes well, but the random split still provided an unfair advantage).
- **Leakage Audit:** Injecting `late_imps` resulted in a biologically impossible **100.0%** precision, validating our rigorous exclusion of overlapping time windows.

## 5. Limitations & Honest Framing

- **Directional Likelihood, Not Deterministic:** Our model provides a directional likelihood (Precision 66% at the top 50 ranks) that a page will experience a traffic drop. It is a decision-support tool, not a crystal ball.
- **Seasonality Blindspot:** The model does not understand temporal intent. It will confidently flag a "Halloween Costumes" page in November as decaying. Human editorial review is required to dismiss seasonal drops.
- **Evergreen Content:** Static pages (like definitions) will trigger high risk due to massive `content_age_days` and low `word_count`, even if they are perfectly stable.

## 6. Ranked Recommendations (The Action Playbook)

Applying the model to the June 2026 dataset, we generated a human-reviewed action queue of **19701** URLs, triaged into three priority buckets:

1. **P1: Striking Distance & Stale -> Full Refresh (2865 URLs):** High traffic, ranking 11-20, and >180 days old. Action: Update facts and expand content to push to Page 1.
2. **P2: High Traffic, Low CTR -> Snippet Optimization (3264 URLs):** Ranking on Page 1 but failing to earn clicks. Action: Rewrite title tags and meta descriptions.
3. **P3: Thin Content -> Expand or Consolidate (13572 URLs):** High risk pages with <500 words. Action: Consolidate via 301 redirects or expand heavily.

## 7. Reproducibility

- **Source Code & Architecture:** All data contracts, baseline heuristic audits, model ablation tests, leakage audits, and the playbook generation scripts are publicly available in the [GitHub Repository](https://github.com/Assem-ElQersh/FlyRank-ML-Internship).
- **Environment:** The repository utilizes a Hugging Face-hosted DuckDB Data Warehouse and standard Python data science libraries (Pandas, Scikit-Learn).

## 8. Acknowledgments & Data Credit

Built on the FlyRank ML Internship dataset. 

Special thanks to the [FlyRank Platform](https://flyrank.ai) for providing the anonymized production search dataset and rigorous engineering standards that made this research possible.